# `gain_skeletons` demonstrator

`gain_skeletons` builds mock `xarray` datasets that scaffold radio
interferometric calibration ("gain") solutions, writes them to zarr, and
reads them back. **It is a demonstrator: every value in every dataset below
is randomly generated, and nothing in this package computes or applies
calibration.**

It ships a catalogue of ten calibration types, split into the
direction-independent ones (`J`, `G`, `T`, opacity, `B`, `D`, antpos,
fringefit) and the direction-dependent ones (generic direction-dependent
gain, ionosphere). The catalogue is illustrative rather than exhaustive:
it exists to cover the range of coordinate shapes these datasets take, and
the last section shows that a type it does not carry needs no change to the
package. This notebook walks through the catalogue using only the package's
public API, so every dataset shown here comes from `gain_skeletons` itself;
the notebook defines no schema of its own.

## Imports and the catalogue

`gain_skeletons` is imported under the conventional short alias `gs`.
`list_cal_types` returns the registry keys, direction-independent types
first.

In [1]:
import numpy as np
import xarray as xr

import gain_skeletons as gs

gs.list_cal_types()

('J',
 'G',
 'T',
 'opacity',
 'B',
 'D',
 'antpos',
 'fringefit',
 'dd_gain',
 'ionosphere')

## Coordinate factories on their own

Each axis the package uses has a standalone factory function that returns a
one-dimensional `xarray.DataArray`. They are usable independently of any
dataset, and their default ranges (a MeerKAT L-band frequency span, a
fixed time origin) are overridable via keyword arguments.

In [2]:
print(gs.time_coord(3, start=0.0, interval=8.0).values)
print(gs.frequency_coord(5).values)
print(gs.frequency_coord(5, start=1.0e9, end=2.0e9).values)
print(gs.antenna_name_coord(4).values)
gs.frequency_coord(4)

[ 0.  8. 16.]
[8.560e+08 1.070e+09 1.284e+09 1.498e+09 1.712e+09]
[1.00e+09 1.25e+09 1.50e+09 1.75e+09 2.00e+09]
['m000' 'm001' 'm002' 'm003']


<xarray.DataArray 'frequency' (frequency: 4)> Size: 32B
array([8.56000000e+08, 1.14133333e+09, 1.42666667e+09, 1.71200000e+09])
Dimensions without coordinates: frequency
Attributes:
    type:                 spectral_coord
    units:                Hz
    observer:             topo
    reference_frequency:  856000000.0
    channel_width:        285333333.3333333

## The simplest case, `G`

The standard electronic gain is a complex, on-diagonal-only gain with one
solution for the whole band. Read the dataset repr below against that
description: four axes, a single complex `GAIN` array, units of `rel`, and a
frequency axis of length one — present, but unresolved, which is not the
same thing as absent.

In [3]:
gs.make_gain_xds("G")

<xarray.Dataset> Size: 752B
Dimensions:         (time: 4, antenna_name: 8, frequency: 1, receptor_label: 2)
Coordinates:
  * time            (time) float64 32B 1.7e+09 1.7e+09 1.7e+09 1.7e+09
  * antenna_name    (antenna_name) <U4 128B 'm000' 'm001' ... 'm006' 'm007'
  * frequency       (frequency) float64 8B 8.56e+08
  * receptor_label  (receptor_label) <U1 8B 'X' 'Y'
Data variables:
    GAIN            (time, antenna_name, frequency, receptor_label) complex64 512B ...
    FLAG            (time, antenna_name, frequency, receptor_label) bool 64B ...
Attributes:
    cal_type:             G
    direction_dependent:  False
    jones_structure:      diagonal
    description:          Standard electronic gain, on-diagonal only, one sol...

## `B` against `G`: channel-resolved against single-channel

`B` is the bandpass: the same axis list as `G`, but resolved per channel
rather than carrying one solution per band. Both are on-diagonal-only
complex gains, so their `GAIN` arrays share the same dimensions; only the
frequency axis's extent differs.

In [4]:
b = gs.make_gain_xds("B", n_frequency=64)
g = gs.make_gain_xds("G")
print("B dims:", dict(b.sizes))
print("G dims:", dict(g.sizes))
print("same axes:", b.GAIN.dims == g.GAIN.dims)

B dims: {'time': 4, 'antenna_name': 8, 'frequency': 64, 'receptor_label': 2}
G dims: {'time': 4, 'antenna_name': 8, 'frequency': 1, 'receptor_label': 2}
same axes: True


## `antpos`: axes genuinely absent, and meaningful parameter labels

`antpos` is neither frequency- nor polarisation-dependent, so it carries no
`frequency` and no `receptor_label` axis whatsoever. An absent axis is
materially different from an axis that exists with length one, such as `G`'s
frequency axis above: the first says the quantity has no such dependence,
the second says it has one solution across that dependence. Its three
same-unit components — an antenna position offset in each of X, Y and Z —
instead occupy a `parameter_label` axis.

In [5]:
antpos = gs.make_gain_xds("antpos")
print("axes present:", antpos.ANTENNA_POSITION_OFFSET.dims)
print("frequency absent:", "frequency" not in antpos.dims)
print("parameter labels:", list(antpos.parameter_label.values))
antpos

axes present: ('time', 'antenna_name', 'parameter_label')
frequency absent: True
parameter labels: [np.str_('dX'), np.str_('dY'), np.str_('dZ')]


<xarray.Dataset> Size: 984B
Dimensions:                  (time: 4, antenna_name: 8, parameter_label: 3)
Coordinates:
  * time                     (time) float64 32B 1.7e+09 1.7e+09 1.7e+09 1.7e+09
  * antenna_name             (antenna_name) <U4 128B 'm000' 'm001' ... 'm007'
  * parameter_label          (parameter_label) <U2 24B 'dX' 'dY' 'dZ'
Data variables:
    ANTENNA_POSITION_OFFSET  (time, antenna_name, parameter_label) float64 768B ...
    FLAG                     (time, antenna_name) bool 32B False False ... False
Attributes:
    cal_type:             antpos
    direction_dependent:  False
    description:          Antenna position correction. Three same-unit compon...

## `ionosphere`: the direction axis

`ionosphere` is one of the two direction-dependent types. `direction` is an
integer index into a direction list held elsewhere (a facet within one MSv4
field of view), not a sky position itself. The axis appears only for
calibration types with genuine direction dependence; none of the
direction-independent types carry it.

In [6]:
gs.make_gain_xds("ionosphere", n_direction=4)

<xarray.Dataset> Size: 1kB
Dimensions:       (direction: 4, time: 4, antenna_name: 8)
Coordinates:
  * direction     (direction) int64 32B 0 1 2 3
  * time          (time) float64 32B 1.7e+09 1.7e+09 1.7e+09 1.7e+09
  * antenna_name  (antenna_name) <U4 128B 'm000' 'm001' 'm002' ... 'm006' 'm007'
Data variables:
    TEC           (direction, time, antenna_name) float64 1kB 0.6287 ... 1.292
    FLAG          (direction, time, antenna_name) bool 128B False ... False
Attributes:
    cal_type:             ionosphere
    direction_dependent:  True
    description:          Ionospheric total electron content. Direction-depen...

## Fringefit both ways

Fringefit is the only calibration type in the catalogue with several
quantities — `PHASE`, `DELAY`, `RATE`, and `DISP_DELAY` — coming from a
single solve, and the only one where its consolidated and split layouts
genuinely differ. `gain_skeletons` offers both, and privileges neither as
correct:

- **Consolidated** (`make_gain_xds`): all four quantities share one
  `PARAMETER` array, indexed by an explicit `parameter_label` axis. This
  keeps every parameter needed to describe one solve adjacent in memory and,
  once written, in one chunked zarr array, and it needs only a single
  `FLAG` to mark a solution bad. The cost is that `DISP_DELAY`, which is
  unpolarised, must be broadcast redundantly across the `receptor_label`
  axis so it can sit in the same array as the three polarised quantities.
- **Split** (`make_split_gain_xds`): each quantity gets its own dataset,
  with exactly the axes it needs — `DISP_DELAY` keeps no `receptor_label`
  at all — and a scalar `units` attribute, since each array has exactly one
  unit. The cost is fragmentation: one solve is spread across four
  datasets, each with its own `FLAG`.

Which is preferable depends on which cost the reader would rather pay:
storing `DISP_DELAY` twice over a `receptor_label` axis it does not have, or
splitting one solve across four datasets with four flags.

In [7]:
consolidated = gs.make_gain_xds("fringefit")
split = gs.make_split_gain_xds("fringefit")

print("consolidated arrays:", list(consolidated.data_vars))
print("split datasets     :", list(split))
consolidated

consolidated arrays: ['PARAMETER', 'FLAG']
split datasets     : ['PHASE', 'DELAY', 'RATE', 'DISP_DELAY']


<xarray.Dataset> Size: 2kB
Dimensions:          (time: 4, antenna_name: 8, frequency: 1,
                      receptor_label: 2, parameter_label: 4)
Coordinates:
  * time             (time) float64 32B 1.7e+09 1.7e+09 1.7e+09 1.7e+09
  * antenna_name     (antenna_name) <U4 128B 'm000' 'm001' ... 'm006' 'm007'
  * frequency        (frequency) float64 8B 8.56e+08
  * receptor_label   (receptor_label) <U1 8B 'X' 'Y'
  * parameter_label  (parameter_label) <U10 160B 'PHASE' ... 'DISP_DELAY'
    parameter_units  (parameter_label) <U3 48B 'deg' 's' 's/s' 's'
Data variables:
    PARAMETER        (time, antenna_name, frequency, receptor_label, parameter_label) float64 2kB ...
    FLAG             (time, antenna_name, frequency, receptor_label) bool 64B ...
Attributes:
    cal_type:             fringefit
    direction_dependent:  False
    description:          Fringe fit. Four quantities with differing units, p...

## Units in the consolidated layout

Because fringefit's four quantities do not share a unit, the consolidated
`PARAMETER` array cannot carry a scalar `units` attribute the way every
other calibration type's array does — that would falsely claim a single
unit for a heterogeneous array. Instead, units move to a `parameter_units`
coordinate aligned with `parameter_label`, which travels along with any
selection.

In [8]:
print("scalar units attr:", consolidated.PARAMETER.attrs.get("units", "<absent>"))
print("parameter_units  :", list(consolidated.parameter_units.values))
print("selecting DELAY  :", consolidated.sel(parameter_label="DELAY").parameter_units.item())
print("split units      :", {k: v[k].attrs["units"] for k, v in split.items()})

scalar units attr: <absent>
parameter_units  : [np.str_('deg'), np.str_('s'), np.str_('s/s'), np.str_('s')]
selecting DELAY  : s
split units      : {'PHASE': 'deg', 'DELAY': 's', 'RATE': 's/s', 'DISP_DELAY': 's'}


## The cost of consolidating

This is the redundancy called out above, made concrete: in the consolidated
layout, `DISP_DELAY`'s values are identical across the `receptor_label`
axis, because the same unpolarised value has been broadcast to both
receptors so it can live in the same array as the polarised quantities.
The split layout never introduces this redundancy — `DISP_DELAY` keeps no
`receptor_label` axis at all — but it does need its own `FLAG`, one per
dataset, where the consolidated layout needs only one.

In [9]:
disp = consolidated.PARAMETER.sel(parameter_label="DISP_DELAY")
print(
    "DISP_DELAY repeated across receptors:",
    np.array_equal(disp.sel(receptor_label="X").values, disp.sel(receptor_label="Y").values),
)
print("consolidated flag dims:", consolidated.FLAG.dims)
print("split flag dims       :", {k: v.FLAG.dims for k, v in split.items()})
print("split DISP_DELAY axes :", split["DISP_DELAY"].DISP_DELAY.dims)

DISP_DELAY repeated across receptors: True
consolidated flag dims: ('time', 'antenna_name', 'frequency', 'receptor_label')
split flag dims       : {'PHASE': ('time', 'antenna_name', 'frequency', 'receptor_label'), 'DELAY': ('time', 'antenna_name', 'frequency', 'receptor_label'), 'RATE': ('time', 'antenna_name', 'frequency', 'receptor_label'), 'DISP_DELAY': ('time', 'antenna_name', 'frequency')}
split DISP_DELAY axes : ('time', 'antenna_name', 'frequency', 'parameter_label')


## Round-trip to zarr and the on-disk layout

Both layouts write to zarr with `consolidated=False` on both write and
read: zarr format 3 does not specify consolidated metadata, so omitting the
flag draws a `ZarrUserWarning` on write and a `RuntimeWarning` on read
(because it hunts for metadata that was never written). Passing
`consolidated=True` on read does not warn — it raises `ValueError` outright,
since it demands metadata that is not there. Everything below lives
in a `tempfile.TemporaryDirectory`, so nothing from this notebook is left
on disk afterwards. The directory listings make the one-array-against-four
difference visible on disk, not just in memory: the consolidated store has
a single `PARAMETER` array directory, while the split stores are four
independent zarr groups.

In [10]:
import tempfile
from pathlib import Path


def show_tree(path: Path, limit: int = 24) -> None:
    """Print the top two levels of a zarr store."""
    entries = sorted(
        p.relative_to(path) for p in path.rglob("*") if len(p.relative_to(path).parts) <= 2
    )
    for entry in entries[:limit]:
        print("  ", entry)
    if len(entries) > limit:
        print(f"   ... and {len(entries) - limit} more")


with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp)

    consolidated.to_zarr(root / "consolidated.zarr", consolidated=False)
    print("consolidated store:")
    show_tree(root / "consolidated.zarr")

    for name, xds in split.items():
        xds.to_zarr(root / f"split_{name}.zarr", consolidated=False)
    print("\nsplit stores:", sorted(p.name for p in root.glob("split_*.zarr")))

    reread = xr.open_dataset(root / "consolidated.zarr", engine="zarr", consolidated=False).load()
    print("\nround-trip identical:", reread.identical(consolidated))

consolidated store:
   FLAG
   FLAG/c
   FLAG/zarr.json
   PARAMETER
   PARAMETER/c
   PARAMETER/zarr.json
   antenna_name
   antenna_name/c
   antenna_name/zarr.json
   frequency
   frequency/c
   frequency/zarr.json
   parameter_label
   parameter_label/c
   parameter_label/zarr.json
   parameter_units
   parameter_units/c
   parameter_units/zarr.json
   receptor_label
   receptor_label/c
   receptor_label/zarr.json
   time
   time/c
   time/zarr.json
   ... and 1 more



split stores: ['split_DELAY.zarr', 'split_DISP_DELAY.zarr', 'split_PHASE.zarr', 'split_RATE.zarr']



round-trip identical: True


## The escape hatch

The ten registered calibration types are a convenience, not a limitation.
`CalSpec` and `ParamSpec` are public, so a calibration type the registry
does not carry — here, an antenna pointing offset — can be hand-written and
passed to `make_gain_xds` exactly like a registry name. The registry is a
catalogue of examples, not the whole of what the package can express.

In [11]:
pointing = gs.CalSpec(
    name="pointing_offset",
    parameters=(
        gs.ParamSpec(
            name="POINTING_OFFSET",
            units="rad",
            axes=("time", "antenna_name", "parameter_label"),
            dtype="float64",
            labels=("dAZ", "dEL"),
            scale=1.0e-4,
        ),
    ),
    default_sizes={"time": 4, "antenna_name": 8},
    description="Antenna pointing correction; not in the registry.",
)

gs.make_gain_xds(pointing)

<xarray.Dataset> Size: 728B
Dimensions:          (time: 4, antenna_name: 8, parameter_label: 2)
Coordinates:
  * time             (time) float64 32B 1.7e+09 1.7e+09 1.7e+09 1.7e+09
  * antenna_name     (antenna_name) <U4 128B 'm000' 'm001' ... 'm006' 'm007'
  * parameter_label  (parameter_label) <U3 24B 'dAZ' 'dEL'
Data variables:
    POINTING_OFFSET  (time, antenna_name, parameter_label) float64 512B 1.257...
    FLAG             (time, antenna_name) bool 32B False False ... False False
Attributes:
    cal_type:             pointing_offset
    direction_dependent:  False
    description:          Antenna pointing correction; not in the registry.